# UFC Medallion Architecture

Bronze → Silver → Gold pipeline for the Kaggle UFC Complete Dataset (all events 1996–2024). The bronze layer ingests raw CSV data directly via `kagglehub`, preserving the source schema with ingestion metadata (`_source_file`, `_ingestion_timestamp`).

In [0]:
pip install kagglehub

In [0]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("maksbasher/ufc-complete-dataset-all-events-1996-2024")

print("Path to dataset files:", path)

In [0]:
import kagglehub
import os

# Download latest version (uses cache if already downloaded)
path = kagglehub.dataset_download("maksbasher/ufc-complete-dataset-all-events-1996-2024")
print("Path to dataset files:", path)

print("\nFiles in dataset (recursive):")
for root, dirs, files in os.walk(path):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        rel_path = os.path.relpath(fpath, path)
        print(f"  {rel_path}  ({size_mb:.2f} MB)")

In [0]:
import pandas as pd

# Examine schemas of all CSV files
csv_files = [
    "UFC dataset/Fighter stats/fighter_stats.csv",
    "UFC dataset/Medium set/medium_dataset.csv",
    "UFC dataset/Large set/large_dataset.csv",
    "UFC dataset/Small set/completed_events_small.csv",
    "UFC dataset/Small set/upcoming_events_small.csv",
]

for csv_file in csv_files:
    full_path = os.path.join(path, csv_file)
    table_name = csv_file.split("/")[-1].replace(".csv", "")
    print(f"\n{'='*60}")
    print(f"Table: {table_name}")
    print(f"Path: {csv_file}")
    pdf = pd.read_csv(full_path)
    print(f"Row count: {len(pdf)}")
    print(f"Columns ({len(pdf.columns)}):")
    for col in pdf.columns:
        print(f"  - {col} ({pdf[col].dtype})")
    print("\nSample (2 rows):")
    print(pdf.head(2).to_string())

In [0]:
from pyspark.sql.types import TimestampType
from pyspark.sql import functions as F
from datetime import datetime

# Create bronze schema
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ufc_bronze")
spark.sql("COMMENT ON SCHEMA workspace.ufc_bronze IS 'Bronze layer: raw UFC data loaded from Kaggle dataset (maksbasher/ufc-complete-dataset-all-events-1996-2024)'")

# Define CSV files to load as bronze tables
csv_files = [
    ("UFC dataset/Fighter stats/fighter_stats.csv", "fighter_stats"),
    ("UFC dataset/Medium set/medium_dataset.csv", "medium_dataset"),
    ("UFC dataset/Large set/large_dataset.csv", "large_dataset"),
    ("UFC dataset/Small set/completed_events_small.csv", "completed_events_small"),
    ("UFC dataset/Small set/upcoming_events_small.csv", "upcoming_events_small"),
]

bronze_tables = []

for csv_rel_path, table_name in csv_files:
    full_path = os.path.join(path, csv_rel_path)
    full_table_name = f"workspace.ufc_bronze.{table_name}"
    
    print(f"\nLoading {table_name}...")
    
    # Read CSV with pandas (driver-local file access)
    pdf = pd.read_csv(full_path)
    
    # Convert to Spark DataFrame
    df = spark.createDataFrame(pdf)
    
    # Add bronze metadata columns
    df = df.withColumn("_source_file", F.lit(csv_rel_path)) \
           .withColumn("_ingestion_timestamp", F.lit(datetime.now()).cast(TimestampType()))
    
    # Write as Delta table (overwrite for idempotency)
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)
    
    row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_table_name}").collect()[0]["cnt"]
    print(f"  -> {full_table_name}: {row_count} rows, {len(df.columns)} columns")
    bronze_tables.append((full_table_name, row_count, len(df.columns)))

print("\n" + "="*60)
print("Bronze layer loaded successfully!")
print(f"{'Table':<45} {'Rows':>8} {'Cols':>6}")
print("-" * 60)
for tname, rows, cols in bronze_tables:
    print(f"{tname:<45} {rows:>8,} {cols:>6}")

In [0]:
# Verify bronze tables - show table list and sample data
print("Bronze tables in workspace.ufc_bronze:")
display(spark.sql("SHOW TABLES IN workspace.ufc_bronze"))

for table_name in ["fighter_stats", "medium_dataset", "large_dataset", "completed_events_small", "upcoming_events_small"]:
    print(f"\n{'='*60}")
    print(f"workspace.ufc_bronze.{table_name}")
    print(f"{'='*60}")
    display(spark.sql(f"SELECT * FROM workspace.ufc_bronze.{table_name} LIMIT 5"))

In [0]:
# ============================================================
# SILVER LAYER — Cleaned & Standardized Data
# ============================================================
from pyspark.sql.functions import *
from pyspark.sql.types import *

SILVER_SCHEMA = "ufc_silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"COMMENT ON SCHEMA {SILVER_SCHEMA} IS 'Silver layer: cleaned and standardized UFC data'")

def bronze_table_exists(table_name):
    try:
        rows = spark.sql("SHOW TABLES IN ufc_bronze").collect()
        return any(r.tableName == table_name for r in rows)
    except Exception:
        return False

# --- Silver: Events (from completed_events_small) ---
if bronze_table_exists("completed_events_small"):
    print("=== Silver: completed_events_small -> events ===")
    df = spark.read.table("ufc_bronze.completed_events_small")
    df = df.drop("_source_file", "_ingestion_timestamp").dropDuplicates()
    for c in df.columns:
        if dict(df.dtypes)[c] == "string":
            df = df.withColumn(c, trim(col(c)))
    if "date" in df.columns:
        df = (df.withColumn("date",
              coalesce(to_date(col("date"), "MMMM d, yyyy"),
                       to_date(col("date"), "yyyy-MM-dd"),
                       to_date(col("date"), "MM/dd/yyyy")))
              .withColumn("event_year", year(col("date")))
              .withColumn("event_month", month(col("date")))
              .withColumn("event_quarter", quarter(col("date"))))
    if "location" in df.columns:
        loc_parts = split(col("location"), ",")
        df = (df
              .withColumn("city", trim(loc_parts.getItem(0)))
              .withColumn("state", when(size(loc_parts) >= 3, trim(loc_parts.getItem(1))).otherwise(lit(None)))
              .withColumn("country", when(size(loc_parts) >= 3, trim(loc_parts.getItem(2))).otherwise(trim(loc_parts.getItem(1)))))
    df.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.events")
    print(f"  Rows: {df.count()}")
    display(df.limit(5))
    print(f"  ✅ {SILVER_SCHEMA}.events created!")
else:
    print("⚠️  completed_events_small not found")

print("\nSilver setup complete.")

In [0]:
# --- Silver: Fights (from large_dataset — main 95-col fight table) ---
if bronze_table_exists("large_dataset"):
    print("=== Silver: large_dataset -> fights ===")
    df = spark.read.table("ufc_bronze.large_dataset")
    df = df.drop("_source_file", "_ingestion_timestamp").dropDuplicates()
    for c in df.columns:
        if dict(df.dtypes)[c] == "string":
            df = df.withColumn(c, trim(col(c)))
    # Resolve winner from "Red"/"Blue" to fighter name
    if "winner" in df.columns and "r_fighter" in df.columns and "b_fighter" in df.columns:
        df = (df.withColumn("winner_name",
              when(col("winner") == "Red", col("r_fighter"))
              .when(col("winner") == "Blue", col("b_fighter"))
              .otherwise(col("winner")))
              .withColumn("winner_is_red",
              when(col("winner") == "Red", True)
              .when(col("winner") == "Blue", False)
              .otherwise(None)))
    # Standardize finish category
    if "method" in df.columns:
        df = (df.withColumn("finish_category",
              when(col("method").rlike("(?i)KO|TKO"), "KO/TKO")
              .when(col("method").rlike("(?i)sub|submission"), "Submission")
              .when(col("method").rlike("(?i)dec|decision"), "Decision")
              .when(col("method").rlike("(?i)DQ|disqual"), "DQ")
              .otherwise(col("method"))))
    df.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.fights")
    print(f"  Rows: {df.count()}")
    display(df.select("event_name", "r_fighter", "b_fighter", "winner_name", "finish_category", "method", "finish_round", "time_sec").limit(5))
    print(f"  ✅ {SILVER_SCHEMA}.fights created!")
else:
    print("⚠️  large_dataset not found")

In [0]:
# --- Silver: Fighters (from fighter_stats) ---
if bronze_table_exists("fighter_stats"):
    print("=== Silver: fighter_stats -> fighters ===")
    df = spark.read.table("ufc_bronze.fighter_stats")
    df = df.drop("_source_file", "_ingestion_timestamp").dropDuplicates()
    for c in df.columns:
        if dict(df.dtypes)[c] == "string":
            df = df.withColumn(c, trim(col(c)))
    # Rename for clarity
    if "height" in df.columns:
        df = df.withColumnRenamed("height", "height_cm")
    if "weight" in df.columns:
        df = df.withColumnRenamed("weight", "weight_kg")
    if "reach" in df.columns:
        df = df.withColumnRenamed("reach", "reach_cm")
    # BMI, win_rate, weight_class
    if "height_cm" in df.columns and "weight_kg" in df.columns:
        df = (df.withColumn("bmi",
              when(col("height_cm").isNotNull() & col("weight_kg").isNotNull() & (col("height_cm") > 0),
                   round(col("weight_kg") / (col("height_cm") / 100) ** 2, 2))
              .otherwise(None)))
    if "wins" in df.columns and "losses" in df.columns:
        df = (df.withColumn("win_rate",
              when((col("wins") + col("losses")) > 0,
                   round(col("wins") / (col("wins") + col("losses")), 4))
              .otherwise(None)))
    if "weight_kg" in df.columns:
        df = (df.withColumn("weight_class",
              when(col("weight_kg") <= 56.7, "Flyweight")
              .when(col("weight_kg") <= 61.2, "Bantamweight")
              .when(col("weight_kg") <= 65.8, "Featherweight")
              .when(col("weight_kg") <= 70.3, "Lightweight")
              .when(col("weight_kg") <= 77.1, "Welterweight")
              .when(col("weight_kg") <= 83.9, "Middleweight")
              .when(col("weight_kg") <= 93.0, "Light Heavyweight")
              .when(col("weight_kg") > 93.0, "Heavyweight")
              .otherwise(None)))
    df.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.fighters")
    print(f"  Rows: {df.count()}")
    display(df.select("name", "wins", "losses", "height_cm", "weight_kg", "reach_cm", "stance", "age", "bmi", "win_rate", "weight_class").limit(5))
    print(f"  ✅ {SILVER_SCHEMA}.fighters created!")
else:
    print("⚠️  fighter_stats not found")

In [0]:
# --- Silver: Fights Medium (from medium_dataset — alternative fight stats) ---
if bronze_table_exists("medium_dataset"):
    print("=== Silver: medium_dataset -> fights_medium ===")
    df = spark.read.table("ufc_bronze.medium_dataset")
    df = df.drop("_source_file", "_ingestion_timestamp").dropDuplicates()
    for c in df.columns:
        if dict(df.dtypes)[c] == "string":
            df = df.withColumn(c, trim(col(c)))
    if "date" in df.columns:
        df = (df.withColumn("date",
              coalesce(to_date(col("date"), "M/d/yyyy"),
                       to_date(col("date"), "MMMM d, yyyy"),
                       to_date(col("date"), "yyyy-MM-dd")))
              .withColumn("event_year", year(col("date")))
              .withColumn("event_month", month(col("date"))))
    if "location" in df.columns:
        loc_parts = split(col("location"), ",")
        df = (df
              .withColumn("city", trim(loc_parts.getItem(0)))
              .withColumn("state", when(size(loc_parts) >= 3, trim(loc_parts.getItem(1))).otherwise(lit(None)))
              .withColumn("country", when(size(loc_parts) >= 3, trim(loc_parts.getItem(2))).otherwise(trim(loc_parts.getItem(1)))))
    if "round" in df.columns:
        df = df.withColumn("round", col("round").cast("int"))
    if "time" in df.columns:
        time_parts = split(col("time"), ":")
        df = (df.withColumn("fight_duration_seconds",
              when(size(time_parts) == 2,
                   time_parts.getItem(0).cast("int") * 60 + time_parts.getItem(1).cast("int"))
              .otherwise(col("time").cast("int"))))
    df.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.fights_medium")
    print(f"  Rows: {df.count()}")
    display(df.select("event", "date", "r_fighter", "b_fighter", "method", "round", "fight_duration_seconds").limit(5))
    print(f"  ✅ {SILVER_SCHEMA}.fights_medium created!")
else:
    print("⚠️  medium_dataset not found")

In [0]:
# ============================================================
# GOLD LAYER — Analytics-Ready Star Schema
# ============================================================
GOLD_SCHEMA = "ufc_gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"COMMENT ON SCHEMA {GOLD_SCHEMA} IS 'Gold layer: analytics-ready UFC fact and dimension tables'")

def silver_table_exists(table_name):
    try:
        rows = spark.sql("SHOW TABLES IN ufc_silver").collect()
        return any(r.tableName == table_name for r in rows)
    except Exception:
        return False

print("Gold schema created.")

In [0]:
# --- Gold: fact_fights (enriched fights with event details) ---
if silver_table_exists("fights"):
    print("=== Building fact_fights ===")
    df_fights = spark.read.table("ufc_silver.fights")
    if silver_table_exists("events"):
        df_events = spark.read.table("ufc_silver.events")
        if "event_name" in df_fights.columns and "event" in df_events.columns:
            df_events_clean = df_events.select(
                col("event"),
                col("date").alias("event_date"),
                col("location").alias("event_location"),
                col("city").alias("event_city"),
                col("state").alias("event_state"),
                col("country").alias("event_country"),
                col("event_year"), col("event_month"), col("event_quarter")
            )
            df_fights = (df_fights.join(df_events_clean, on=col("event_name") == col("event"), how="left"))
            print("  Joined with events on event_name = event")
    df_fights = df_fights.withColumn("fight_id", monotonically_increasing_id())
    df_fights.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.fact_fights")
    print(f"  Rows: {df_fights.count()}")
    display(df_fights.select("fight_id", "event_name", "r_fighter", "b_fighter", "winner_name", "finish_category", "method", "finish_round", "time_sec", "event_date", "event_location").limit(5))
    print(f"  ✅ {GOLD_SCHEMA}.fact_fights created!")
else:
    print("⚠️  ufc_silver.fights not found")

In [0]:
# --- Gold: dim_fighter ---
if silver_table_exists("fighters"):
    print("=== Building dim_fighter ===")
    df_dim = spark.read.table("ufc_silver.fighters").withColumn("fighter_key", monotonically_increasing_id())
    df_dim.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.dim_fighter")
    print(f"  Rows: {df_dim.count()}")
    display(df_dim.select("fighter_key", "name", "wins", "losses", "height_cm", "weight_kg", "reach_cm", "stance", "age", "bmi", "win_rate", "weight_class").limit(5))
    print(f"  ✅ {GOLD_SCHEMA}.dim_fighter created!")
else:
    print("⚠️  ufc_silver.fighters not found")